# EDA — le label : taux d'admission par cellule

**Session de travail du 29 août 2026.**

La question à laquelle ce carnet répond, et une seule :

> Le taux d'admission par cellule `(formation × session × type de bac × boursier)`
> est-il prévisible, et à partir de quelles variables ?

Tout ce qui ne sert pas cette question n'a pas sa place ici. J'explore d'abord la
structure du fichier, puis la cible, puis seulement les variables explicatives —
regarder les variables avant la cible reviendrait à chercher sans savoir quoi.

Chaque résultat est suivi de ce que j'en conclus pour la modélisation. Un
graphique sans phrase est un graphique inutile.

In [ ]:
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, str(Path.cwd().parent / "src"))
from edumatch.config import load_settings

# L'environnement `dev` ne déclare que les deux derniers millésimes, pour itérer
# vite pendant le développement. Une analyse exploratoire a besoin de la période
# complète : c'est la seule façon de voir les ruptures de série. Je charge donc
# explicitement la configuration qui déclare les huit sessions.
settings = load_settings("prod")
RAW = settings.raw_dir / "parcoursup"
MILLESIMES = settings.donnees.parcoursup.millesimes


def charger(annee: int) -> pl.DataFrame:
    """Lit un millésime brut. Le fichier porte un BOM UTF-8, que Polars retire."""
    return pl.read_csv(
        RAW / f"parcoursup_{annee}.csv",
        separator=";",
        encoding="utf8-lossy",
        infer_schema_length=2000,
    )


print("millésimes déclarés en configuration :", MILLESIMES)

## 1. Forme et types

Premier réflexe : savoir ce que je manipule, avant toute interprétation.

In [ ]:
df_2025 = charger(2025)
print("forme 2025 :", df_2025.shape)
df_2025.head(3).select(df_2025.columns[:6])

**Ce que j'en conclus.** 14 252 lignes et 118 colonnes pour la session 2025. Le
volume est modeste — une centaine de mégaoctets pour les huit sessions — et tient
intégralement en mémoire. C'est ce qui justifie de réserver un moteur distribué à
la base Sirene, dont l'ordre de grandeur est tout autre, et de traiter ce
catalogue avec un moteur en mémoire.

## 2. Le grain : que représente une ligne ?

C'est la question à trancher avant toute autre. Tant que je ne sais pas ce qu'est
une ligne, je ne peux rien affirmer sur les données. Je cherche donc la clé : la
combinaison de colonnes qui identifie une ligne et une seule.

In [ ]:
candidates = [
    ["cod_uai"],
    ["cod_uai", "fili"],
    ["cod_uai", "lib_for_voe_ins"],
    ["cod_aff_form"],
]
for cles in candidates:
    n = df_2025.select(cles).n_unique()
    marque = "  <-- clé" if n == df_2025.height else ""
    print(f"{'+'.join(cles):<30} {n:>7} uniques sur {df_2025.height}{marque}")

print()
print("cod_aff_form — valeurs manquantes :", df_2025["cod_aff_form"].null_count())

**Ce que j'en conclus.** `cod_aff_form`, le code d'affectation de la formation,
identifie une ligne et une seule : 14 252 valeurs distinctes, aucune manquante.

Le grain est donc : **une ligne = une formation, identifiée par son code
d'affectation, pour une session donnée**. Les huit millésimes empilés, la clé
devient `(session, cod_aff_form)`.

L'établissement seul ne suffit pas — 4 058 établissements pour 14 252 formations,
un même établissement en proposant plusieurs. C'est ce qui explique que mes
cellules de label soient définies au niveau de la formation, et non de
l'établissement.

## 3. Colonnes sans information

Une colonne dont toutes les valeurs sont identiques ne peut rien expliquer.

In [ ]:
for col in df_2025.columns:
    if df_2025[col].n_unique() <= 1:
        vides = df_2025[col].null_count()
        valeur = df_2025[col].unique().to_list()[:1]
        print(f"{col:<28} valeur unique={valeur}  vides={vides}/{df_2025.height}")

**Ce que j'en conclus.** `session` est constante par construction : il y a un
fichier par millésime. Elle ne portera de l'information qu'une fois les huit
empilés, où elle deviendra ma dimension temporelle.

Les deux colonnes `*_id_paysage` sont vides à 100 %. Je ne conclus pas tout de
suite qu'elles sont inutiles : je vérifie d'abord leur comportement sur les huit
millésimes. Une colonne vide en 2025 peut avoir été renseignée avant.

## 4. Stabilité du schéma entre millésimes

Mon protocole d'évaluation est temporel : entraînement sur 2018-2023, validation
sur 2024, test sur 2025. Une variable qui change de disponibilité entre ces
périodes est un piège — le modèle apprendrait sur une information qui n'existera
plus au moment de la prédiction.

In [ ]:
dfs = {a: charger(a) for a in MILLESIMES}
colonnes = {a: set(d.columns) for a, d in dfs.items()}

communes = set.intersection(*colonnes.values())
union = set.union(*colonnes.values())

print("colonnes par millésime :", {a: len(c) for a, c in colonnes.items()})
print()
print("présentes dans les 8 millésimes :", len(communes))
print("vues au moins une fois          :", len(union))
print("donc instables                  :", len(union) - len(communes))

**Ce que j'en conclus.** Le fichier s'est enrichi au fil des ans : 85 colonnes en
2018, 118 depuis 2021. Seules **83 colonnes sont présentes sur les huit
sessions** — c'est le socle sur lequel un modèle entraîné sur toute la période
peut s'appuyer.

Les 45 autres ne sont pas inutilisables, mais elles imposent un choix explicite :
restreindre la période d'entraînement, ou accepter de les traiter comme absentes
sur les millésimes anciens. Je tranche au moment de construire les variables, et
je consigne la décision.

In [ ]:
taux = {
    col: {a: 1 - dfs[a][col].null_count() / dfs[a].height for a in MILLESIMES}
    for col in sorted(communes)
}

toujours_pleines = sum(1 for t in taux.values() if min(t.values()) > 0.99)
print(f"colonnes communes remplies à plus de 99 % partout : {toujours_pleines}/{len(communes)}")
print()

instables = sorted(
    (
        (max(t.values()) - min(t.values()), col, t)
        for col, t in taux.items()
        if max(t.values()) - min(t.values()) > 0.10
    ),
    reverse=True,
)
for ecart, col, t in instables:
    detail = "  ".join(f"{a}:{t[a] * 100:>3.0f}%" for a in MILLESIMES)
    print(f"{col:<18} écart {ecart * 100:>5.1f} pts   {detail}")

**Ce que j'en conclus.** 59 colonnes sur 83 sont remplies à plus de 99 % sur toute
la période : un socle sain, qui me dispense d'une stratégie d'imputation lourde.

Trois colonnes bougent, et il faut distinguer deux phénomènes très différents :

- `pct_etab_orig` passe de 46 % à 100 % **en 2023**. Ce n'est pas une saisie qui
  s'améliore progressivement, c'est une rupture nette — donc un changement de
  règle de publication.
- `acc_term` et `acc_term_f` restent autour de 45 à 56 % sur toute la période. Ce
  n'est pas une rupture, c'est une caractéristique permanente du fichier.

Ces deux cas appellent des décisions différentes. Je les examine séparément.

## 5. Le manque est-il aléatoire ?

C'est la question qui compte, plus que le taux de manque lui-même. Une valeur
manquante distribuée au hasard est du bruit, que l'on peut imputer. Une valeur
manquante dont l'absence dépend d'une autre variable est une **information** — et
l'imputer reviendrait à détruire cette information tout en fabriquant de la
donnée.

In [ ]:
d = df_2025.with_columns(pl.col("acc_term").is_null().alias("manquant"))
resume = (
    d.group_by("fili")
    .agg(pl.len().alias("n"), pl.col("manquant").mean().alias("taux_manquant"))
    .filter(pl.col("n") >= 100)
    .sort("taux_manquant")
)
for filiere, n, tx in resume.iter_rows():
    print(f"{filiere[:32]:<32} n={n:>5}   manquant = {tx * 100:>5.1f} %")

**Ce que j'en conclus, et c'est un résultat important.**

Zéro ou cent pour cent. Jamais entre les deux.

`acc_term` n'est publié que pour les BTS et les CPGE. Ce n'est donc pas une
valeur *manquante*, c'est une valeur **non applicable** : son absence est
entièrement déterminée par la filière.

Trois conséquences directes sur ma modélisation :

1. **Imputer par la moyenne serait absurde.** Je fabriquerais une valeur pour
   100 % des licences à partir d'une moyenne calculée uniquement sur des BTS et
   des CPGE. Ce serait inventer de la donnée.
2. **L'absence est elle-même porteuse d'information.** Un modèle à qui je donne
   « `acc_term` est absent » apprend en réalité « cette formation n'est ni un BTS
   ni une CPGE » : je lui redonnerais la filière sous une autre forme, sans
   l'avoir voulu.
3. **C'est une décision, pas un nettoyage.** Trois options se défendent : écarter
   la colonne, la retenir uniquement là où elle est définie, ou la retenir avec un
   indicateur explicite de non-applicabilité. Ce qui ne se défend pas, c'est de
   combler les trous sans le dire.

In [ ]:
for annee in (2022, 2023):
    d = dfs[annee].with_columns(pl.col("pct_etab_orig").is_null().alias("manquant"))
    resume = (
        d.group_by("fili")
        .agg(pl.len().alias("n"), pl.col("manquant").mean().alias("taux_manquant"))
        .filter(pl.col("n") >= 200)
        .sort("taux_manquant")
    )
    print(f"--- {annee} : pct_etab_orig, taux de manque par filière")
    for filiere, n, tx in resume.iter_rows():
        print(f"    {filiere[:28]:<28} n={n:>5}   manquant = {tx * 100:>5.1f} %")
    print()

**Ce que j'en conclus.** Même mécanisme que `acc_term`, puis élargissement :
jusqu'en 2022, `pct_etab_orig` n'était publié que pour les BTS et les CPGE ; à
partir de 2023, il l'est pour toutes les filières.

Ce n'est donc pas une donnée dont la qualité s'améliore, c'est une **règle de
publication qui change**. Pour un protocole d'évaluation temporel, la conséquence
est nette : sur 2018-2022, cette variable ne décrit que deux filières ; sur
2023-2025, elle les décrit toutes. Un modèle entraîné sur la période mixte
apprendrait une régularité qui n'est plus vraie au moment du test.

In [ ]:
for col in ("etablissement_id_paysage", "composante_id_paysage"):
    ligne = []
    for a in MILLESIMES:
        if col in dfs[a].columns:
            renseignees = dfs[a].height - dfs[a][col].null_count()
            ligne.append(f"{a}: {renseignees:>5}/{dfs[a].height}")
        else:
            ligne.append(f"{a}: absente")
    print(col)
    print("   " + "   ".join(ligne))
    print()

**Ce que j'en conclus.** Ces deux colonnes ne sont pas « vides » : elles sont
**mortes**. Absentes jusqu'en 2020, renseignées à environ 47 % de 2021 à 2024,
puis totalement vides en 2025.

C'est le scénario qui dégrade un modèle en silence : entraîné sur une période où
la variable existe, évalué sur une période où elle a disparu, sans qu'aucune
erreur ne soit levée.

Je les écarte — mais je tiens à la formulation exacte de la raison, parce que
c'est elle qui se défend : non pas « ces colonnes sont vides », mais **« ces
colonnes sont instables entre ma période d'entraînement et ma période de
test »**. La première est un constat, la seconde est un argument.

## Bilan de la reconnaissance

| Point examiné | Ce que j'ai établi |
|---|---|
| Forme | 14 252 × 118 pour la session 2025 |
| Grain | une ligne = `(session, cod_aff_form)`, clé vérifiée, sans doublon ni manque |
| Socle stable | 83 colonnes présentes sur les 8 sessions, dont 59 remplies à plus de 99 % |
| Manque non aléatoire | `acc_term` : 0 % ou 100 % selon la filière — non applicable, pas manquant |
| Rupture de série | `pct_etab_orig` : publication élargie à toutes les filières en 2023 |
| Colonnes mortes | deux colonnes renseignées de 2021 à 2024, vides en 2025 |

Trois faits de dérive relevés dans les données elles-mêmes, avant toute
modélisation. Ils alimenteront directement le dispositif de surveillance : je sais
désormais que la dérive n'est pas une hypothèse d'école sur ce jeu de données.

**La suite** : construire la cible et regarder sa distribution, avant de toucher
aux variables explicatives.

---

# Phase 2 — la cible

Je construis maintenant le label, et je le regarde **avant** de toucher aux
variables explicatives. L'ordre n'est pas indifférent : explorer 118 colonnes sans
savoir ce que l'on cherche à expliquer, c'est produire des graphiques au hasard.

Ma cellule d'analyse est `(formation × session × type de bac × boursier)`, et le
taux se calcule ainsi :

$$\text{taux} = \frac{\texttt{prop\_tot}\_{bg|bt|bp}[\_brs]}{\texttt{nb\_voe\_pp}\_{bg|bt|bp}[\_brs]}$$

Soit six cellules par formation : trois types de baccalauréat, chacun décliné
« tous candidats » et « boursiers ».

In [ ]:
CELLULES = ["bg", "bg_brs", "bt", "bt_brs", "bp", "bp_brs"]

entete = f"{'cellule':<10}{'définies':>10}{'part':>8}{'taux>1':>9}{'taux=0':>9}{'taux=1':>9}{'moyenne':>10}{'médiane':>10}"
print(entete)
for suffixe in CELLULES:
    num, den = f"prop_tot_{suffixe}", f"nb_voe_pp_{suffixe}"
    d = df_2025.filter(pl.col(den) > 0).with_columns((pl.col(num) / pl.col(den)).alias("taux"))
    n = d.height
    print(
        f"{suffixe:<10}{n:>10}{n / df_2025.height * 100:>7.1f}%"
        f"{(d['taux'] > 1).sum():>9}{(d['taux'] == 0).sum():>9}{(d['taux'] == 1).sum():>9}"
        f"{d['taux'].mean():>10.3f}{d['taux'].median():>10.3f}"
    )

**Ce que j'en conclus.** Le nombre de cellules exploitables décroît régulièrement :
96 % des formations ont une cellule « bac général », 83,6 % seulement une cellule
« bac professionnel boursier ». C'est attendu — certaines formations ne reçoivent
aucun vœu d'un profil donné — mais cela signifie que **mes six cellules ne
couvrent pas la même population**, et qu'une comparaison directe entre elles
demande de la prudence.

Le point qui m'arrête, en revanche, est la colonne `taux>1` : **1 219 cellules
dépassent 1 pour le seul bac général**. Un taux d'admission supérieur à 100 % n'a
pas de sens en l'état. Je dois comprendre avant de corriger.

In [ ]:
d_bg = df_2025.filter(pl.col("nb_voe_pp_bg") > 0).with_columns(
    (pl.col("prop_tot_bg") / pl.col("nb_voe_pp_bg")).alias("taux")
)

print("bac général — 13 685 cellules définies")
print(f"  moyenne brute          : {d_bg['taux'].mean():.4f}")
print(f"  moyenne bornée à 1     : {d_bg['taux'].clip(0, 1).mean():.4f}")
print(f"  moyenne en excluant >1 : {d_bg.filter(pl.col('taux') <= 1)['taux'].mean():.4f}")
print()

print("le dépassement disparaît-il sur les grosses cellules ?")
print(f"{'seuil de vœux':>16}{'cellules':>12}{'part':>8}{'dont taux>1':>13}")
for seuil in (1, 10, 30, 100):
    g = d_bg.filter(pl.col("nb_voe_pp_bg") >= seuil)
    print(f"{seuil:>16}{g.height:>12}{g.height / d_bg.height * 100:>7.1f}%{(g['taux'] > 1).sum():>13}")

**Ce que j'en conclus, et c'est le résultat central de cette analyse.**

Le dépassement **ne disparaît pas** quand j'élimine les petites cellules : à 100
vœux minimum, 483 cellules sur 7 154 dépassent encore 1. Ce n'est donc pas un
artefact de petits effectifs, c'est **structurel**.

L'explication tient à ce que comptent réellement mes deux termes. `nb_voe_pp`
compte des **vœux**, c'est-à-dire des candidats. `prop_tot` compte des
**propositions émises**, c'est-à-dire des événements : une formation de 100 places
émet bien davantage que 100 propositions au fil de la campagne, puisque chaque
désistement libère une place et déclenche une nouvelle proposition.

Numérateur et dénominateur ne décrivent donc pas la même population. Borner à 1
n'est pas « corriger des valeurs aberrantes » — c'est **imposer une borne
sémantique** à un rapport qui n'est pas un taux au sens strict. C'est défendable,
à condition de le dire ainsi.

Je note au passage que la moyenne de 0,522 que je retenais jusqu'ici correspond
exactement à la **moyenne bornée** : la borne était déjà appliquée dans mon
calcul, mais elle n'était écrite nulle part. Elle l'est désormais.

In [ ]:
d = df_2025.filter(pl.col("nb_voe_pp_bg") > 0)
print(f"{'définition':<36}{'>1':>7}{'part>1':>9}{'moyenne':>10}{'médiane':>10}")
alternatives = [
    ("prop_tot / nb_voe_pp  (retenue)", "prop_tot_bg", "nb_voe_pp_bg"),
    ("prop_tot / nb_cla_pp", "prop_tot_bg", "nb_cla_pp_bg"),
    ("acc / nb_voe_pp", "acc_bg", "nb_voe_pp_bg"),
    ("acc / nb_cla_pp", "acc_bg", "nb_cla_pp_bg"),
]
for nom, num, den in alternatives:
    x = d.filter(pl.col(den) > 0).with_columns((pl.col(num) / pl.col(den)).alias("t"))
    print(
        f"{nom:<36}{(x['t'] > 1).sum():>7}{(x['t'] > 1).mean() * 100:>8.1f}%"
        f"{x['t'].mean():>10.3f}{x['t'].median():>10.3f}"
    )

**Ce que j'en conclus — et pourquoi je n'ai pas pris la solution facile.**

La définition `acc / nb_voe_pp` ne dépasse jamais 1 : trois cellules sur 13 685.
Elle réglerait le problème d'un trait. **Je l'écarte pourtant**, et c'est un
arbitrage que je dois savoir défendre.

`acc` compte les candidats qui ont **accepté la proposition et se sont inscrits**.
Or un lycéen qui reçoit une proposition et la décline parce qu'il a mieux ailleurs
a bel et bien été *admis*. En retenant `acc`, je ne mesurerais plus
l'accessibilité de la formation, mais un mélange entre sa sélectivité et **les
préférences des candidats** : une formation facile d'accès mais peu désirée
obtiendrait un score bas.

Mon système promet une chose précise — *quelles sont mes chances d'être admis* —
et non *vais-je m'y inscrire*. La définition retenue est donc celle qui répond à
la question posée, même si elle est moins commode.

### Les quatre décisions que j'arrête ici

1. **Conserver `prop_tot / nb_voe_pp`** : seule définition qui mesure
   l'accessibilité et non la préférence du candidat.
2. **Borner le taux à 1**, en énonçant la raison : le rapport est interprété comme
   une probabilité d'admission, et le dépassement provient de propositions
   réémises après désistement.
3. **Pondérer par l'effectif de la cellule** à l'entraînement : une cellule de
   trois vœux ne peut pas peser autant qu'une cellule de cinq cents.
4. **Ne pas exclure les petites cellules** : la pondération traite déjà leur
   moindre fiabilité, et un seuil à trente vœux écarterait 26 % des observations —
   perte inacceptable au regard du bénéfice.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
libelles = {"bg": "Bac général", "bt": "Bac technologique", "bp": "Bac professionnel"}

for ax, suffixe in zip(axes, ["bg", "bt", "bp"]):
    num, den = f"prop_tot_{suffixe}", f"nb_voe_pp_{suffixe}"
    taux = (
        df_2025.filter(pl.col(den) > 0)
        .with_columns((pl.col(num) / pl.col(den)).clip(0, 1).alias("taux"))["taux"]
        .to_list()
    )
    ax.hist(taux, bins=50, color="#4C72B0", edgecolor="white", linewidth=0.4)
    ax.set_title(f"{libelles[suffixe]} — n={len(taux)}")
    ax.set_xlabel("taux d'admission (borné à 1)")
    ax.grid(axis="y", alpha=0.25)

axes[0].set_ylabel("nombre de cellules")
fig.suptitle("Distribution du taux d'admission par cellule — session 2025", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print(f"{'cellule':<8}{'=0':>8}{'part':>8}{'=1':>8}{'part':>8}{'moyenne':>10}{'médiane':>10}")
for suffixe in ["bg", "bt", "bp"]:
    num, den = f"prop_tot_{suffixe}", f"nb_voe_pp_{suffixe}"
    t = (
        df_2025.filter(pl.col(den) > 0)
        .with_columns((pl.col(num) / pl.col(den)).clip(0, 1).alias("taux"))["taux"]
    )
    zeros, uns = (t == 0).sum(), (t == 1).sum()
    print(
        f"{suffixe:<8}{zeros:>8}{zeros / len(t) * 100:>7.1f}%{uns:>8}{uns / len(t) * 100:>7.1f}%"
        f"{t.mean():>10.3f}{t.median():>10.3f}"
    )

**Ce que j'en conclus.** La distribution n'est pas gaussienne, et cela oriente
directement mes choix de modélisation.

Elle est **étalée sur tout l'intervalle**, avec deux masses aux bornes : des
cellules à 0 (aucune proposition émise à ce profil) et des cellules à 1 (tous les
vœux ont reçu une proposition). Ces deux masses ne sont pas du bruit : elles
correspondent à des situations réelles — une formation très sélective qui ne prend
aucun bachelier professionnel, une formation en tension nulle qui accepte tout le
monde.

Trois conséquences pour la suite :

- une **erreur absolue moyenne pondérée** est plus lisible qu'une erreur
  quadratique sur ce type de cible bornée ;
- la **calibration** devra être vérifiée explicitement : un modèle peut avoir une
  bonne erreur moyenne tout en plaçant mal les valeurs extrêmes, or ce sont
  précisément elles qui intéressent un candidat ;
- le décalage entre **moyenne et médiane** confirme l'asymétrie de la
  distribution, et interdit de résumer la sélectivité par une seule moyenne.

La progression entre les trois baccalauréats est nette et sera à quantifier à
l'étape suivante : c'est le cœur de l'enjeu d'équité du projet.